---
# IMPORTS

In [2]:
import sys, os
sys.path.insert(0, os.path.join('..'))   # project root on path

import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import wrds
import polars as pl
import polars.selectors as cs
import pyarrow
import config
import password

from src.data_loading import load_crsp, load_futures, load_crsp_polars, wrds_fetch, load_cz_monthly, Fama_French_fetch
from src.preprocessing import clean_crsp, clean_futures
from src.feature_engineering import add_target, add_volatility_momentum, crosssectional_rank, get_feature_cols, polars_features
from src.utils import generate_batches, train_val_test_split, split_batch, shrink, shrink_polars

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})
pd.set_option('display.float_format', '{:.4f}'.format)

---
# Load daily dataset and Feature engineering

In [3]:

data = pd.read_parquet(config.CRSP_PATH_CLEAN)
data = data.sort_values(['PERMNO', 'date']).reset_index(drop=True)
data['date'] = pd.to_datetime(data['date'])
print(data.shape)
print(data.columns.tolist())
print(data.dtypes)
print(data.head())


(19603583, 4)
['PERMNO', 'date', 'ret', 'mkt_ret']
PERMNO              int32
date       datetime64[ms]
ret               float64
mkt_ret           float64
dtype: object
   PERMNO       date     ret  mkt_ret
0   10001 2000-01-03  0.0074  -0.0095
1   10001 2000-01-04 -0.0146  -0.0383
2   10001 2000-01-05  0.0148   0.0019
3   10001 2000-01-06 -0.0073   0.0010
4   10001 2000-01-07 -0.0074   0.0271


---
# Build features

This part of the code handle the generation of features from the returns column. It also shrink the data type of the dataframe in order to save RAM.

In [10]:
from src.feature_engineering import polars_features

df = polars_features(data, value_col='ret', date_col='date', target_col = 'ret', reversal=True)

print(f'Dataset RAM size before shrink : {df.estimated_size() / 1024**3:.2f}Gb')
df = shrink_polars(df)
print(f'Dataset RAM size after shrink : {df.estimated_size() / 1024**3:.2f}Gb')
 

df = df.to_pandas()
df.head()

df.to_parquet(config.FEATURES_PATH_CLEAN, compression='zstd')

Dataset RAM size before shrink : 2.73Gb
Dataset RAM size after shrink : 1.49Gb


In [ ]:
from src.feature_engineering import build_features

"""

To run once, DO NOT RUN if features.parquet already exists in /data/processed

"""

# 1. Load data and shrink

data = pd.read_parquet(config.CRSP_PATH_CLEAN)
data = data.sort_values(['PERMNO', 'date']).reset_index(drop=True)
data['date'] = pd.to_datetime(data['date'])
print(data.shape)
print(data.columns.tolist())
print(f'Dataset RAM size before shrink : {data.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
data = shrink(data)
print(f'Dataset RAM size after shrink : {data.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
print(data.dtypes)
print(data.head())


"""

build_features function creates the following :

- 1 day reversal
- Momentum on multiple windows  
- Momentum scaled by volatility on multiple windows
- Volatility on multiple windows

"""

# 2. Build features

df = build_features(data, config.VALUE_RETURN, trading_interval=False)
 
print(df.head())


# 3. Shrink and upload to .parquet

print(f'Dataset RAM size before shrink : {df.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
df = shrink(df)
print(f'Dataset RAM size after shrink : {df.memory_usage(index=True).sum() / 1024**3:.2f}Gb')


df.to_parquet(config.FEATURES_PATH_CLEAN, compression='zstd')

---
# Load Chen-Zimmerman Dataset

In [9]:
from src.data_loading import load_cz_monthly


# 1. Load and shrink

cz_daily = load_cz_monthly()

print(f'Dataset RAM size before shrink : {cz_daily.memory_usage(index=True).sum() / 1024**3:.2f}Gb')
cz_daily = shrink(cz_daily)
print(f'Dataset RAM size after shrink : {cz_daily.memory_usage(index=True).sum() / 1024**3:.2f}Gb')


# 2. Index cz dataset on date

cz_indexed = cz_daily.set_index('date').sort_index()
cz_indexed.index = pd.to_datetime(cz_indexed.index).astype('datetime64[ms]') # Prepare to merge


# 3. Upload to .parquet 

cz_indexed.to_parquet(config.CZ_PATH_CLEAN, compression='zstd')
cz_indexed.head()

Initial Size of the dataset: (1140, 206)
Total Date range : 1926-01-30 00:00:00 -> 2020-12-31 00:00:00
Dataset shape after dropping column with less than 90.0% completion: (1140, 57)
Total column dropped so far : 149
Dataset shape after dropping highly correlated (corr_coef > 0.95) columns: (1140, 55)
Total column dropped so far : 151
Dataset RAM size before shrink : 0.01Gb
Dataset RAM size after shrink : 0.01Gb


,Beta,BetaFP,BetaTailRisk,BidAskSpread,CompEquIss,Coskewness,DivInit,DivOmit,DivSeason,DivYieldST,...,Size,Spinoff,std_turn,STreversal,VolMkt,VolSD,VolumeTrend,zerotrade,zerotradeAlt1,zerotradeAlt12
date,,,,,,,,,,,,,,,,,,,,,
1932-01-30,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856
1932-01-31,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856
1932-02-01,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856
1932-02-02,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856
1932-02-03,0.1656,0.2146,0.0289,0.2590,0.1298,-0.0455,-0.0021,-0.0090,0.0077,-0.0327,...,0.0841,0.0722,-0.0608,0.4873,0.0002,0.0180,0.0907,0.0462,0.0798,0.0856


---
# Generate Batch and Merge

Generating the random sampled batch and merging the other dataset on the batch. 

We merge other dataset with the daily dataset only at the batch level in order to largely decrease the RAM usage duriung the operation considering the size of the daily dataframe with around 6700 stocks.

In [ ]:
features = pd.read_parquet(config.FEATURES_PATH_CLEAN)
cz = pd.read_parquet(config.CZ_PATH_CLEAN)
print(f'Number of unique PERMNO in dataset : {features['PERMNO'].nunique()}')

split_batch_dict = split_batch(features, 'PERMNO', 'date', df_to_join=[cz], batch_number=config.BATCH_NUMBER, batch_size=config.BATCH_SIZE, overlap=False)

---
# Fama-French 3 Factors

In [ ]:
# Start by loading the data
permno_list = features['PERMNO'].unique()


# Start and End date are taken as min and max of initial dataset as for now the ff3 is for benchmark purpose


ff3_df = Fama_French_fetch(permno_list = permno_list, start_date = features['date'].min(), end_date = features['date'].max())


---
# Handling NaN due to merge/join

As we merged the different dataset, we use how='left' in order to use the daily dataset as the baseline. However, due to differences in data range among datasets, we will have to handle NaN after merging.

This is a Work In Progress !